[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_04_Building_Your_First_Agent.ipynb)

# 🤖 Lesson 4: Building Your First Agent — The ReAct Loop

**Series:** Becoming a Practical AI/LLM Engineer  
**Prerequisites:** Lessons 1–3 (LLM fundamentals, prompt engineering, tool use)

---

## What You'll Build Today

By the end of this lesson you will have built a **working AI agent** — one that can:
- Think through a problem step by step
- Decide which tools to use and when
- Handle tool failures and correct itself
- Know when it has enough information to stop

This is the moment the curriculum shifts from *"calling an LLM"* to *"building an agent"*. Those are fundamentally different things.

---

## Section 1: What Is an Agent, Really?

In Lesson 1 you made API calls. In Lesson 3 you used tool calling. You might wonder: "Isn't that already an agent?"

Not quite. Here's the distinction:

| Concept | What it is |
|---|---|
| **LLM call** | One-shot: you send a prompt, you get a response. Done. |
| **Tool use** | One-shot with a side-effect: the model calls a function, you get a result. Still done after one round-trip (or a few). |
| **Agent** | A **loop**: the model reasons → acts → observes → reasons again → acts → … until a goal is reached. The model decides how many steps to take. |

The key word is **loop**. An agent runs in a cycle driven by the model's own reasoning. You (the developer) don't tell it "call tool A, then tool B". The model figures that out.

### Why does the loop change everything?

With a loop, the agent can:
- **Plan**: break a big goal into sub-steps
- **Recover from failure**: if a tool returns an error, reason about it and try again
- **Chain actions**: use the output of one tool as input to another
- **Know when to stop**: decide "I have enough information, here's my final answer"

---

## Section 2: The ReAct Pattern — Reasoning + Acting

**ReAct** is the most important and widely-used agent pattern. It was introduced in a 2022 paper (*"ReAct: Synergizing Reasoning and Acting in Language Models"*) and is the foundation of nearly every production agent you'll encounter.

The pattern has four alternating steps:

```
┌─────────────────────────────────────────┐
│                                         │
│   Thought → Action → Observation        │
│       ↑                    │            │
│       └────────────────────┘            │
│         (loop until done)               │
│                  ↓                      │
│           Final Answer                  │
│                                         │
└─────────────────────────────────────────┘
```

### What each step means:

1. **Thought** — The model narrates its reasoning. "I need to find the weather first, then check if the user's city has any alerts."
2. **Action** — The model calls a tool (function call). `get_weather(city="Mumbai")`
3. **Observation** — The tool result comes back. `{"temp": 34, "condition": "sunny"}`
4. **Back to Thought** — The model incorporates the observation and reasons about what to do next.

This cycle repeats until the model decides it has a complete answer.

### Why "Thought" matters

The Thought step is what separates ReAct from naive tool-calling. When the model writes out its reasoning *before* acting, it:
- Makes better decisions about which tool to call
- Catches mistakes before they propagate
- Produces more reliable, traceable behavior

This is chain-of-thought (from Lesson 2) applied to action planning.

---

## Section 3: How ReAct Maps to the Anthropic API

Here's the critical insight: **the Anthropic tool-use API is ReAct under the hood.**

| ReAct concept | Anthropic API equivalent |
|---|---|
| Thought | The text content in the assistant's response (before tool calls) |
| Action | A `tool_use` block in the assistant's response |
| Observation | A `tool_result` block you send back in the user turn |
| Final Answer | An assistant response with `stop_reason: "end_turn"` and no tool calls |

The **agent loop** is the code you write that keeps calling the API until `stop_reason == "end_turn"` with no tool calls.

Let's build it.

---
## ⚙️ Setup

In [ ]:
# Install the Anthropic SDK
!pip install anthropic -q

import anthropic
import json
import time

# Load your API key from Colab Secrets
# Go to: 🔑 (left sidebar) → Add new secret → Name: ANTHROPIC_API_KEY → Value: your key
from google.colab import userdata
API_KEY = userdata.get('ANTHROPIC_API_KEY')

client = anthropic.Anthropic(api_key=API_KEY)
print("✅ Anthropic client ready")

---
## Section 4: Define the Agent's Tools

Our agent will be a **Research Assistant** that can:
- Search the web (simulated)
- Do math calculations
- Check the current date/time
- Save notes

We'll use simulated (fake) tool implementations so you don't need any external APIs. The *structure* is what matters — you could swap in real implementations later.

### Step 1: Write the tool implementations (Python functions)

In [ ]:
# ============================================================
# TOOL IMPLEMENTATIONS (the actual Python functions)
# ============================================================

import math
from datetime import datetime

# Simulated search results database
SEARCH_DATABASE = {
    "anthropic": "Anthropic is an AI safety company founded in 2021 by Dario Amodei, Daniela Amodei, and others. They created Claude, a family of AI assistants. Anthropic raised $7.3B in funding and is valued at ~$18B as of 2024.",
    "openai": "OpenAI is an AI research company known for GPT-4 and ChatGPT. Founded in 2015, it became one of the most valuable AI companies with products used by over 100 million people.",
    "react pattern": "ReAct (Reasoning + Acting) is a prompting framework for LLM agents introduced by Yao et al. (2022). It interleaves chain-of-thought reasoning with tool calls, enabling more reliable and interpretable agent behavior.",
    "python": "Python is a high-level programming language known for simplicity and wide adoption in data science, AI/ML, web development, and automation. It has a rich ecosystem with libraries like numpy, pandas, and PyTorch.",
    "java": "Java is a statically typed, object-oriented language. It's known for 'write once, run anywhere' via the JVM and is widely used in enterprise backend systems.",
    "llm": "Large Language Models (LLMs) are neural networks trained on massive text datasets. They can generate text, answer questions, write code, and perform many language tasks. Examples: GPT-4, Claude, Gemini, Llama.",
    "vector database": "A vector database stores high-dimensional embeddings and enables semantic similarity search. Examples: Pinecone, Weaviate, Chroma. Essential for RAG (Retrieval-Augmented Generation) systems.",
}

notes_storage = []  # In-memory note storage


def search_web(query: str) -> str:
    """Simulated web search. Returns facts about the query topic."""
    query_lower = query.lower()
    
    # Check each keyword in our database
    for keyword, result in SEARCH_DATABASE.items():
        if keyword in query_lower:
            return f"Search result for '{query}':\n{result}"
    
    return f"No results found for '{query}'. Try a different search term."


def calculate(expression: str) -> str:
    """Evaluates a mathematical expression safely."""
    try:
        # Only allow safe math operations
        allowed_names = {k: v for k, v in math.__dict__.items() if not k.startswith('_')}
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"Result of '{expression}' = {result}"
    except Exception as e:
        return f"Error evaluating '{expression}': {str(e)}"


def get_current_time() -> str:
    """Returns the current date and time."""
    now = datetime.now()
    return f"Current time: {now.strftime('%Y-%m-%d %H:%M:%S')} (local)"


def save_note(content: str) -> str:
    """Saves a note for later retrieval."""
    note_id = len(notes_storage) + 1
    notes_storage.append({"id": note_id, "content": content, "timestamp": datetime.now().isoformat()})
    return f"Note #{note_id} saved successfully: '{content[:50]}...'"


def get_notes() -> str:
    """Retrieves all saved notes."""
    if not notes_storage:
        return "No notes saved yet."
    notes_text = "\n".join([f"#{n['id']}: {n['content']}" for n in notes_storage])
    return f"Saved notes:\n{notes_text}"


# ============================================================
# TOOL REGISTRY — maps tool name → Python function
# This is how the agent loop knows which function to call
# ============================================================
TOOL_REGISTRY = {
    "search_web": search_web,
    "calculate": calculate,
    "get_current_time": get_current_time,
    "save_note": save_note,
    "get_notes": get_notes,
}

print("✅ Tool implementations ready")
print(f"Available tools: {list(TOOL_REGISTRY.keys())}")

### Step 2: Write the tool schemas (what the LLM sees)

Remember from Lesson 3: the LLM doesn't see your Python code. It sees JSON schemas that describe what each tool does and what parameters it takes.

In [ ]:
# ============================================================
# TOOL SCHEMAS — what the LLM sees in the API call
# ============================================================

TOOLS = [
    {
        "name": "search_web",
        "description": "Search the web for information about a topic. Use this when you need facts, definitions, or background knowledge about something.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query. Be specific for better results. Example: 'Anthropic company overview'"
                }
            },
            "required": ["query"]
        }
    },
    {
        "name": "calculate",
        "description": "Evaluate a mathematical expression. Use this for any arithmetic, algebra, or math operations. Do not try to compute math in your head — use this tool.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A Python math expression to evaluate. Examples: '2 ** 10', '(18 * 7) / 3', 'sqrt(144)'"
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "get_current_time",
        "description": "Get the current date and time. Use this when the user asks about the current time or date, or when you need to timestamp something.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "save_note",
        "description": "Save a note or piece of information for later reference. Use this to record important findings during research.",
        "input_schema": {
            "type": "object",
            "properties": {
                "content": {
                    "type": "string",
                    "description": "The content to save as a note."
                }
            },
            "required": ["content"]
        }
    },
    {
        "name": "get_notes",
        "description": "Retrieve all previously saved notes. Use this to recall information you saved earlier in the conversation.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
]

print(f"✅ {len(TOOLS)} tool schemas defined")

---
## Section 5: The Agent Loop — The Heart of the Agent

This is the most important code in this lesson. Read every line carefully.

The loop does exactly three things in a cycle:
1. **Call the LLM** with the current conversation history
2. **Execute any tools** the LLM requested
3. **Append results** to the conversation history and go back to step 1

It stops when the LLM says `stop_reason == "end_turn"` (meaning: "I'm done, I have my answer").

```
messages = [user message]
        │
        ▼
   ┌─── LLM call ───────────────────────────┐
   │                                         │
   │  stop_reason == "tool_use"?             │
   │         │ YES                           │
   │         ▼                               │
   │   Execute tool(s)                       │
   │   Append results to messages            │
   │   Loop back ─────────────────────────── ┘
   │
   │  stop_reason == "end_turn"?
   │         │ YES
   │         ▼
   │   Return final response ✅
   │
   └───────────────────────────────────────────
```

In [ ]:
# ============================================================
# THE AGENT LOOP
# ============================================================

def execute_tool(tool_name: str, tool_input: dict) -> str:
    """
    Looks up the tool in the registry and calls it with the given input.
    Returns the result as a string (or an error message if the tool fails).
    """
    if tool_name not in TOOL_REGISTRY:
        return f"Error: Unknown tool '{tool_name}'. Available tools: {list(TOOL_REGISTRY.keys())}"
    
    try:
        tool_fn = TOOL_REGISTRY[tool_name]
        result = tool_fn(**tool_input)  # Call the Python function with the LLM's arguments
        return str(result)
    except Exception as e:
        # IMPORTANT: Always catch errors and return them as strings.
        # The agent can then reason about the failure and try a different approach.
        return f"Error calling tool '{tool_name}': {str(e)}"


def run_agent(user_message: str, max_steps: int = 10, verbose: bool = True) -> str:
    """
    Runs the ReAct agent loop.
    
    Args:
        user_message: The user's request
        max_steps: Safety limit — stop after this many LLM calls (prevents infinite loops)
        verbose: If True, print each step so you can watch the agent think
    
    Returns:
        The agent's final answer as a string
    """
    
    # ------------------------------------------------------------------
    # SYSTEM PROMPT — shapes the agent's personality and behavior
    # ------------------------------------------------------------------
    system_prompt = """You are a helpful research assistant with access to tools.

When answering questions:
1. Think about what you need to do step by step
2. Use your tools to gather real information — do not guess or make things up
3. If a tool call fails, read the error and try a different approach
4. Once you have enough information, provide a clear, well-structured final answer

Always prefer using tools over relying on your training knowledge for factual questions."""
    
    # ------------------------------------------------------------------
    # CONVERSATION HISTORY — this is what grows as the agent loops
    # ------------------------------------------------------------------
    messages = [
        {"role": "user", "content": user_message}
    ]
    
    if verbose:
        print("=" * 60)
        print(f"🧑 USER: {user_message}")
        print("=" * 60)
    
    step = 0
    
    # ------------------------------------------------------------------
    # THE LOOP
    # ------------------------------------------------------------------
    while step < max_steps:
        step += 1
        
        if verbose:
            print(f"\n--- Step {step} ---")
        
        # STEP A: Call the LLM
        response = client.messages.create(
            model="claude-opus-4-5",
            max_tokens=2048,
            system=system_prompt,
            tools=TOOLS,
            messages=messages
        )
        
        # STEP B: Append the assistant's response to history
        # (We always do this, whether there are tool calls or not)
        messages.append({"role": "assistant", "content": response.content})
        
        # STEP C: Print the assistant's thoughts (text before any tool calls)
        if verbose:
            for block in response.content:
                if block.type == "text" and block.text.strip():
                    print(f"\n💭 THOUGHT: {block.text}")
        
        # STEP D: Check the stop reason
        if response.stop_reason == "end_turn":
            # The agent is done! Extract the final text response.
            final_answer = ""
            for block in response.content:
                if block.type == "text":
                    final_answer += block.text
            
            if verbose:
                print(f"\n{'=' * 60}")
                print(f"✅ FINAL ANSWER (after {step} step(s)):")
                print(f"{'=' * 60}")
                print(final_answer)
            
            return final_answer
        
        # STEP E: If stop_reason == "tool_use", execute the tool(s)
        if response.stop_reason == "tool_use":
            tool_results = []  # Collect results from ALL tools called in this step
            
            for block in response.content:
                if block.type == "tool_use":
                    tool_name = block.name
                    tool_input = block.input
                    tool_use_id = block.id
                    
                    if verbose:
                        print(f"\n🔧 ACTION: {tool_name}({json.dumps(tool_input)})")
                    
                    # Execute the tool
                    result = execute_tool(tool_name, tool_input)
                    
                    if verbose:
                        print(f"👁️  OBSERVATION: {result[:200]}{'...' if len(result) > 200 else ''}")
                    
                    # Package the result in the format Anthropic expects
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": tool_use_id,  # Must match the ID from the tool_use block
                        "content": result
                    })
            
            # STEP F: Add all tool results back into the conversation as a user message
            # This is the "Observation" step in ReAct — feeding results back to the model
            messages.append({
                "role": "user",
                "content": tool_results
            })
            
            # Loop back to call the LLM again with the updated history
            continue
        
        # Safety: unknown stop reason
        print(f"⚠️  Unexpected stop_reason: {response.stop_reason}. Stopping.")
        break
    
    return "Agent reached maximum steps without a final answer."


print("✅ Agent loop defined — ready to run!")

---
## Section 6: Watch the Agent Think

Now let's actually run the agent and watch the ReAct loop in action. You'll see each Thought, Action, and Observation printed in real time.

### Example 1: A simple multi-step research task

In [ ]:
# 💡 EXPERIMENT: Change this question to anything you like!
# Try: "What is Anthropic and what is 2^10?"
# Try: "Search for ReAct pattern and save a summary note"
# Try: "What time is it and what is 7 * 8 + 42?"

result = run_agent(
    "What is Anthropic? Also, what is 1234 * 5678? Show me both answers.",
    verbose=True
)

### What just happened?

Look at the output above. You should see:
- **💭 THOUGHT**: The model's internal reasoning before it acts
- **🔧 ACTION**: The tool call(s) it made
- **👁️ OBSERVATION**: What the tool returned
- The loop repeating for each tool
- **✅ FINAL ANSWER**: Once the model has everything it needs

This is ReAct. Thought → Action → Observation → Thought → ... → Final Answer.

You didn't tell the agent "search first, then calculate". It figured that out itself.

### Example 2: A task requiring multiple sequential steps

In [ ]:
# This task requires: search → calculate → save_note → get_notes
# Watch how many steps the agent takes!

# 💡 EXPERIMENT: Can you predict how many steps this will take before running?

result = run_agent(
    """I need you to:
    1. Look up what an LLM is
    2. Calculate what 2 to the power of 16 equals (this is related to token counts)
    3. Save a brief note summarizing both facts
    4. Finally, retrieve my notes and show them to me""",
    verbose=True
)

### Example 3: Self-correction when a tool fails

This is one of the most powerful agent behaviors — what happens when a tool returns an error or unexpected result? A good agent tries a different approach rather than giving up.

In [ ]:
# This query won't match any keyword in our fake database
# Watch what the agent does when search returns "No results found"

# 💡 EXPERIMENT: Notice how the agent handles the missing result.
# Does it give up? Try different search terms? Use its own knowledge?
# The behavior depends on the system prompt — try modifying it and see what changes.

result = run_agent(
    "Search for information about 'transformer architecture' and give me a summary.",
    verbose=True
)

---
## Section 7: Dissecting the Agent State

Let's add a helper to inspect the full conversation history after a run — this is what the LLM actually sees at each step.

In [ ]:
# ============================================================
# DEBUG VERSION: shows the full message history
# ============================================================

def run_agent_debug(user_message: str) -> list:
    """Like run_agent but also returns the full message history for inspection."""
    system_prompt = """You are a helpful research assistant with access to tools.
Use your tools to gather information, then provide a clear final answer."""
    
    messages = [{"role": "user", "content": user_message}]
    
    for step in range(10):
        response = client.messages.create(
            model="claude-opus-4-5",
            max_tokens=1024,
            system=system_prompt,
            tools=TOOLS,
            messages=messages
        )
        
        messages.append({"role": "assistant", "content": response.content})
        
        if response.stop_reason == "end_turn":
            break
        
        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = execute_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })
            messages.append({"role": "user", "content": tool_results})
    
    return messages


# Run a simple query and inspect the raw message history
history = run_agent_debug("What is Python? Quick answer.")

print(f"\n📜 Full message history ({len(history)} messages):")
print("=" * 60)
for i, msg in enumerate(history):
    role = msg['role'].upper()
    content = msg['content']
    
    if isinstance(content, str):
        print(f"\n[Message {i}] {role}: {content[:150]}")
    elif isinstance(content, list):
        for block in content:
            if hasattr(block, 'type'):
                if block.type == 'text':
                    print(f"\n[Message {i}] {role} (text): {block.text[:150]}")
                elif block.type == 'tool_use':
                    print(f"\n[Message {i}] {role} (tool_use): {block.name}({block.input})")
            elif isinstance(block, dict):
                if block.get('type') == 'tool_result':
                    print(f"\n[Message {i}] {role} (tool_result): {str(block.get('content', ''))[:150]}")

### Key insight from the debug output

The agent's "memory" within a task is just this `messages` list. Every time you loop, you pass the *entire conversation history* to the LLM. The LLM doesn't have internal state — it reads the whole conversation from scratch each time and figures out what to do next.

This has implications:
- Long agent runs consume a lot of tokens (the history grows)
- The context window is the limit on how long an agent can run
- Managing this memory is exactly what **Lesson 5** covers

---
## Section 8: Self-Correction — Making Your Agent Resilient

The agent above already handles errors gracefully (errors return as strings, the model reasons about them). But let's make it *explicitly* retry on failure with a different approach.

In [ ]:
# ============================================================
# RESILIENT AGENT: explicit retry logic + step counting
# ============================================================

def run_agent_with_retry(user_message: str, max_steps: int = 15) -> dict:
    """
    Enhanced agent that tracks:
    - How many steps it took
    - Which tools were called
    - Whether any errors occurred
    """
    system_prompt = """You are a careful research assistant.

Rules:
- Always use tools for factual questions — do not guess
- If a tool returns an error or no results, try rephrasing the query or using a different tool
- If you've tried 3 times and still can't find information, say so honestly
- Break complex tasks into individual tool calls — don't try to do too much at once"""
    
    messages = [{"role": "user", "content": user_message}]
    
    # Tracking metadata
    steps_taken = 0
    tools_called = []
    errors_encountered = []
    
    while steps_taken < max_steps:
        steps_taken += 1
        
        response = client.messages.create(
            model="claude-opus-4-5",
            max_tokens=2048,
            system=system_prompt,
            tools=TOOLS,
            messages=messages
        )
        
        messages.append({"role": "assistant", "content": response.content})
        
        if response.stop_reason == "end_turn":
            final_text = "".join(b.text for b in response.content if hasattr(b, 'text'))
            return {
                "answer": final_text,
                "steps": steps_taken,
                "tools_called": tools_called,
                "errors": errors_encountered
            }
        
        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    tools_called.append(block.name)
                    result = execute_tool(block.name, block.input)
                    
                    # Track errors
                    if result.startswith("Error") or "No results found" in result:
                        errors_encountered.append(f"{block.name}: {result[:80]}")
                    
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })
            
            messages.append({"role": "user", "content": tool_results})
    
    return {
        "answer": "Max steps reached without completion.",
        "steps": steps_taken,
        "tools_called": tools_called,
        "errors": errors_encountered
    }


# Test it with a complex multi-step question
# 💡 EXPERIMENT: Try asking something that requires multiple tool calls
result = run_agent_with_retry(
    "Compare Anthropic and OpenAI briefly, then calculate how many days are in 7 years (assuming 365 days/year), and tell me what time it is."
)

print("\n📊 Agent Run Summary:")
print(f"  Steps taken:   {result['steps']}")
print(f"  Tools called:  {result['tools_called']}")
print(f"  Errors:        {result['errors'] or 'None'}")
print(f"\n📝 Answer:")
print(result['answer'])

---
## Section 9: 💡 Experiments — Break Things and Learn

The best way to understand agents is to poke at them. Try each of these:

In [ ]:
# ============================================================
# EXPERIMENT A: What happens with no tools?
# ============================================================
# The LLM falls back to its training knowledge.
# Compare this output to when the agent has tools.

# 💡 EXPERIMENT: Comment out one line at a time and see what changes

no_tools_response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=512,
    system="You are a helpful assistant.",
    # tools=TOOLS,   # <-- Commented out! No tools available
    messages=[{"role": "user", "content": "What is Anthropic? And what is 1234 * 5678?"}]
)

print("Response WITHOUT tools:")
print(no_tools_response.content[0].text)
print(f"\nstop_reason: {no_tools_response.stop_reason}")

In [ ]:
# ============================================================
# EXPERIMENT B: Observe the token cost of the agent loop
# ============================================================
# Each loop iteration sends the ENTIRE history to the API.
# This shows you why token management matters.

# 💡 EXPERIMENT: Run this with a longer task and watch token counts grow

system_prompt = "You are a helpful assistant. Use tools when needed."
messages = [{"role": "user", "content": "Search for 'java', then search for 'python', then calculate 100 * 200"}]
total_input_tokens = 0
total_output_tokens = 0
step = 0

while step < 10:
    step += 1
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=1024,
        system=system_prompt,
        tools=TOOLS,
        messages=messages
    )
    
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    total_input_tokens += input_tokens
    total_output_tokens += output_tokens
    
    print(f"Step {step}: input_tokens={input_tokens:,}, output_tokens={output_tokens:,} | Running total: {total_input_tokens + total_output_tokens:,}")
    
    messages.append({"role": "assistant", "content": response.content})
    
    if response.stop_reason == "end_turn":
        break
    
    if response.stop_reason == "tool_use":
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = execute_tool(block.name, block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
        messages.append({"role": "user", "content": tool_results})

print(f"\n💰 Total tokens used: {total_input_tokens + total_output_tokens:,}")
print(f"   Input:  {total_input_tokens:,} | Output: {total_output_tokens:,}")
print("\n🔑 Insight: Notice how input_tokens GROWS each step because we resend the full history!")
print("   This is why Lesson 5 (Agent Memory) matters — we need strategies to manage this.")

In [ ]:
# ============================================================
# EXPERIMENT C: Add a new tool yourself!
# ============================================================
# Let's add a 'unit_convert' tool and plug it into the agent.
# This shows you how extensible the pattern is.

# Step 1: Write the Python implementation
def unit_convert(value: float, from_unit: str, to_unit: str) -> str:
    """Converts between common units."""
    conversions = {
        ("km", "miles"): 0.621371,
        ("miles", "km"): 1.60934,
        ("kg", "lbs"): 2.20462,
        ("lbs", "kg"): 0.453592,
        ("celsius", "fahrenheit"): None,  # Special case
        ("fahrenheit", "celsius"): None,  # Special case
    }
    
    key = (from_unit.lower(), to_unit.lower())
    
    if key == ("celsius", "fahrenheit"):
        result = (value * 9/5) + 32
        return f"{value}°C = {result:.2f}°F"
    elif key == ("fahrenheit", "celsius"):
        result = (value - 32) * 5/9
        return f"{value}°F = {result:.2f}°C"
    elif key in conversions:
        result = value * conversions[key]
        return f"{value} {from_unit} = {result:.4f} {to_unit}"
    else:
        return f"Conversion from {from_unit} to {to_unit} not supported. Supported: km↔miles, kg↔lbs, celsius↔fahrenheit"


# Step 2: Register it
TOOL_REGISTRY["unit_convert"] = unit_convert

# Step 3: Add the schema
TOOLS.append({
    "name": "unit_convert",
    "description": "Convert between units of measurement. Supports: km↔miles, kg↔lbs, celsius↔fahrenheit",
    "input_schema": {
        "type": "object",
        "properties": {
            "value": {"type": "number", "description": "The numeric value to convert"},
            "from_unit": {"type": "string", "description": "Source unit (e.g., 'km', 'kg', 'celsius')"},
            "to_unit": {"type": "string", "description": "Target unit (e.g., 'miles', 'lbs', 'fahrenheit')"}
        },
        "required": ["value", "from_unit", "to_unit"]
    }
})

print(f"✅ Tool added! Now have {len(TOOLS)} tools: {[t['name'] for t in TOOLS]}")

# Step 4: Use it!
# 💡 EXPERIMENT: Try asking about conversions in different ways
result = run_agent(
    "How far is 100 km in miles? And what is 37 degrees Celsius in Fahrenheit?",
    verbose=True
)

---
## Section 10: Safety Guardrails — Why `max_steps` Matters

A critical lesson in production agent engineering: **always have a step limit**.

Without `max_steps`, an agent can get stuck in a loop:
- Tool returns error → model tries again → same error → tries again → ...
- Ambiguous goal → model keeps searching for more information → never decides to stop
- Model misinterprets tool output → incorrect reasoning → more tool calls → ...

In [ ]:
# ============================================================
# SAFETY: Test max_steps with a VERY low limit
# ============================================================
# This demonstrates what happens when the limit kicks in early

# 💡 EXPERIMENT: Try max_steps=1, 2, 3 and see how partial completion looks

print("Running with max_steps=2 (artificially low):")
result = run_agent(
    "Search for Anthropic, then search for OpenAI, then calculate 10 + 20.",
    max_steps=2,
    verbose=True
)

print("\n---")
print("💡 Key takeaways about max_steps:")
print("  • In production: set max_steps to 10–25 depending on task complexity")
print("  • Track step count in your monitoring/logging")
print("  • Alert if agents routinely hit max_steps (suggests prompt or tool issues)")
print("  • A well-designed agent should complete most tasks in < 5 steps")

---
## 🎓 Lesson Summary

You just built a complete AI agent from scratch. Here's what you now understand:

### The ReAct Loop
```
User Message
    ↓
LLM call (with tools + full history)
    ↓
stop_reason == "tool_use"?  →  Execute tool(s)  →  Append results  →  Loop
stop_reason == "end_turn"?  →  Return final answer  ✅
```

### Three things that make an agent:
1. **A loop** — code that calls the LLM repeatedly until done
2. **Tools** — Python functions the LLM can invoke
3. **History** — the growing `messages` list the LLM reads each iteration

### What you built:
- ✅ A tool registry (5 tools + you added a 6th)
- ✅ A ReAct agent loop with verbose output
- ✅ Error handling and self-correction
- ✅ Token cost tracking
- ✅ Safety guardrails (max_steps)

### What's next — Lesson 5: Agent Memory

You saw in Experiment B that **token counts grow every step**. For a 10-step agent run with large tool results, you can easily burn 50,000+ tokens — just on one task. Lesson 5 covers:

- **Short-term memory**: The context window and how to manage it
- **Summarization**: Compress old conversation turns to save tokens
- **Long-term memory**: Persisting information across agent runs (files, databases)
- **Episodic memory**: Letting an agent "remember" past interactions

---

## 📚 Deepen Your Understanding

If you want to go deeper on what you learned today:

- **Original ReAct paper**: [arxiv.org/abs/2210.03629](https://arxiv.org/abs/2210.03629) — very readable, 8 pages
- **Anthropic tool use docs**: [docs.anthropic.com/en/docs/tool-use](https://docs.anthropic.com/en/docs/tool-use)
- **Try**: Replace the fake `search_web` with a real API (e.g., Tavily, Brave Search) — same schema, different implementation

---

*Lesson 4 of the "Becoming a Practical AI/LLM Engineer" series*  
*Next: Lesson 5 — Agent Memory (short-term, long-term, episodic)*